In [3]:
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [170]:
base_url = 'https://raw.githubusercontent.com/londonaicentre/datatakehome/refs/heads/main/data'
patients =  pd.read_csv(os.path.join(base_url,'patients.csv'))

# Data Quality Issues - patients.csv

1. Duplicated patient ids, with some holding different data. (blocking)
2. Patient first and last names in a ilogical format. (non-blocking)
3. Implausible Birth and Death Dates (non-blocking)
4. Inconsistent ssn format

## 1. Duplicated patient identifiers with mismatching demographics data

#### Issue:
Blocking downstream analytics. 
- When duplicated rows join with other tables, the duplicates can multiply record counts down the pipeline, leading to incorrect agggregated metrics such as averages and counts.
- Conflicting information in duplicated rows can cause the reporting of conflicting results down the pipeline. 

#### Approach:

1) Drop duplicated rows
2) Investigate for mismatch patterns in remaining duplicated rows to design data cleaning approach
  - mismatch of birthdate and format
  - additional numerical suffix after first, last and maiden names
  - duplicated records all have '999 Duplicate Street' as address
3) Remove numerical suffix from column 'first' , 'last' , 'maiden'
4) For the rest of the duplicated patient ids, flag the ones with address '999 Duplicated Road' as DQ and exclude from downstream analysis. 

In [98]:
print(f'Total patient id count: {patients.shape[0]}')
print(f'Unique patient id count: {patients["Id"].nunique()}')
print(f'Total patient id count (after removing duplicating rows): {patients.drop_duplicates().shape[0]}')

Total patient id count: 1182
Unique patient id count: 1171
Total patient id count (after removing duplicating rows): 1182


In [102]:
# drop duplicates
patients.drop_duplicates(inplace=True)

# get duplicated ids with conflicting demographics data
duplicated_pt_ids = patients[patients['Id'].duplicated()]['Id']

# visualise examples
patients[patients['Id'].isin(duplicated_pt_ids)].reset_index().sort_values(by=['Id', 'index']).head(4)

,index,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
9,1001,278c5ad9-1e68-45f5-b274-0bdbb60569be,1969-04-13,NaN,999-85-4871,S99979506,X21577942X,Mrs.,Joanna347,Hermiston71,NaN,Rolfson709,M,white,nonhispanic,F,Arlington Massachusetts US,105 Murphy Dam,East Longmeadow,Massachusetts,Hampden County,NaN,42.091905,-72.523353,1131869.75,8580.43
14,1174,278c5ad9-1e68-45f5-b274-0bdbb60569be,1969-12-30 00:00:00,NaN,999-85-4871,S99979506,X21577942X,Mrs.,Joanna34783,Hermiston71872,NaN,Rolfson709,M,white,nonhispanic,F,Arlington Massachusetts US,999 Duplicate Street,East Longmeadow,Massachusetts,Hampden County,NaN,42.091905,-72.523353,1131869.75,8580.43
1,132,355b4981-d84e-4055-96db-956464714214,1998-10-18,NaN,999-42-5355,S99921573,X58400598X,Ms.,Viola977,McGlynn426,NaN,NaN,NaN,black,nonhispanic,F,Oxford Massachusetts US,848 Murray Viaduct Apt 39,Boston,Massachusetts,Suffolk County,2125.0,42.237403,-71.116778,30017.39,332.14
12,1172,355b4981-d84e-4055-96db-956464714214,1999-06-20 00:00:00,NaN,999-42-5355,S99921573,X58400598X,Ms.,Viola977515,McGlynn426942,NaN,NaN,NaN,black,nonhispanic,F,Oxford Massachusetts US,999 Duplicate Street,Boston,Massachusetts,Suffolk County,2125.0,42.237403,-71.116778,30017.39,332.14


## 2. Presence of implausible birth dates that are in the future / after death date

#### Issue

Non-blocking issue in this analytical pipeline, as birthdate and deathdate are not required for this analysis.

#### Approach
1) Flag as DQ, but can still be included for downstream analysis

In [ ]:
# convert date columns to datetime format
patients['BIRTHDATE'] = pd.to_datetime(patients['BIRTHDATE'].str[:10])
patients['DEATHDATE']  = pd.to_datetime(patients['DEATHDATE'])

print(f'Total count of birth > date date: {patients[patients["BIRTHDATE"] > patients["DEATHDATE"]].shape[0]}')

print(f'Total count of future birth date: {patients[patients["BIRTHDATE"]>pd.Timestamp.now()].shape[0]}')
print(f'Total count of future death date: {patients[patients["DEATHDATE"]>pd.Timestamp.now()].shape[0]}')

Total count of birth > date date: 28
Total count of future birth date: 23
Total count of future death date: 0


In [165]:
print('Examples of birthdate > deathdate :')

# visualise examples
patients[['Id','BIRTHDATE','DEATHDATE']][patients['BIRTHDATE']>patients['DEATHDATE']].head(2)

Examples of birthdate > deathdate :


,Id,BIRTHDATE,DEATHDATE
23,2c634692-05d8-4122-9f46-dd9402e2cfc6,1970-12-22,1963-12-24
56,1d7fe784-8667-47e8-b733-8e6ec0fbb5f9,2074-03-12,2006-01-08


In [169]:
print('Examples of future birthdate : ')

# visualise examples
patients[['Id','BIRTHDATE']][(patients["BIRTHDATE"]>pd.Timestamp.now())].head(2)

Examples of future birthdate : 


,Id,BIRTHDATE
47,301ebb70-0078-4989-b5dd-d751f56c404d,2075-02-18
56,1d7fe784-8667-47e8-b733-8e6ec0fbb5f9,2074-03-12


## 3. Presence of invalid SSNs. 

#### Issue:
The social security number is a 9-digit number that comes in the format of XXX-XX-XXXX, however some patients' SSN are in an invalid format, using a mixture of letters and special characters.

Non-blocking issue as patient_id can also be used as an unique identifier of each patient. The SSN is not a required data field in this analysis pipeline.

#### Approach
1) flag as DQ to raise awareness but record can still be used in downstream analysis. 

In [155]:
correct_ssn = patients["SSN"].str.match(r"\d{3}-\d{2}-\d{4}")
print(f'Total count of incorrect SSN: {patients[~correct_ssn].shape[0]}')

# examples of invalid ssn
patients[~correct_ssn][['Id', 'SSN']].head(2)

Total count of incorrect SSN: 23


,Id,SSN
35,d49f748f-928d-40e8-92c8-73e4c5679711,[|!tyLBh- (hO
70,cdef53d5-0537-4073-8874-79fa5e0346e6,#w#n=M>#:


## Low granularity for individual components of birth address

#### Issue

- The BIRTHPLACE column stores the full address in one string.
- Storing the full address in one string can cause difficulty in filtering data, spotting inconsistent data formats.
- Non-blocking in the context of this analytic pipeline as the birth address is not a required field. 

### Approach

- Separate address components into separate columns. 

In [162]:
# examples
patients[['Id','BIRTHPLACE']].head(2)

,Id,BIRTHPLACE
0,1d604da9-9a81-4ba9-80c2-de3375d59b40,Marigot Saint Andrew Parish DM
1,034e9e3b-2def-4559-bb2a-7850888ae060,Danvers Massachusetts US
